# 03. Analisis exploratorio de las imagenes

Este cuaderno estudia las fotografias como fuente de datos: describe sus propiedades tecnicas, sus
atributos fotometricos, la presencia de imagenes repetidas, y las diferencias de las condiciones de
captura entre temporadas y entre los conjuntos de entrenamiento y prueba. Se apoya en la tabla de
atributos que produce `src.caracteristicas` y en los metadatos guardados por el cuaderno 01.

## Entorno

In [ ]:
import sys
from pathlib import Path


def raiz_proyecto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "src").is_dir():
            return candidato
    raise RuntimeError("No se encontro la raiz del proyecto")


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

%matplotlib inline
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src import caracteristicas, carga, graficos, tablas
from src.config import ORDEN_DANOS, UMBRAL_NITIDEZ

graficos.aplicar_estilo()

## Construccion de la tabla de atributos

Analizar las imagenes requiere convertir cada archivo en un vector de atributos medibles. El modulo
`src.caracteristicas` recorre las fotografias en paralelo y guarda el resultado en una cache de
Parquet, de modo que los analisis posteriores no vuelven a abrir los archivos originales. El proceso
es reanudable: si ya existe la cache, la construccion la reutiliza. Cada imagen se reduce a una
miniatura antes de calcular los atributos fotometricos, porque medidas como la nitidez dependen de
la resolucion y sin normalizar reflejarian el tamano del archivo en vez del contenido de la escena.

In [ ]:
rutas = caracteristicas.listar_rutas_particion("train") + caracteristicas.listar_rutas_particion("test")
atributos = caracteristicas.construir_tabla(rutas, reanudar=True, guardar=True)
print(f"imagenes descritas: {len(atributos)}")
print(f"imagenes con error de lectura: {int(atributos['error'].notna().sum())}")

## Union con las etiquetas

Se combinan los metadatos de entrenamiento y prueba en un solo conjunto de etiquetas y se unen con
la tabla de atributos por la ruta relativa de la imagen, con una combinacion interna para conservar
solo las imagenes que tienen tanto etiqueta como atributos calculados. El resultado se separa en un
subconjunto de entrenamiento y uno de prueba.

In [ ]:
metadatos_train = carga.cargar_metadatos("train")
metadatos_test = carga.cargar_metadatos("test")
etiquetas = pd.concat([metadatos_train, metadatos_test], ignore_index=True)

datos = etiquetas.merge(atributos, on="ruta_relativa", how="inner", suffixes=("", "_img"))
entrenamiento = datos[datos["particion"] == "train"].copy()
prueba = datos[datos["particion"] == "test"].copy()

print(f"conjunto combinado: {len(datos)} imagenes")
print(f"entrenamiento: {len(entrenamiento)} | prueba: {len(prueba)}")

## Propiedades tecnicas de los archivos

Se describen las dimensiones, la relacion de aspecto, los megapixeles y el peso de los archivos, las
resoluciones mas frecuentes, la proporcion de imagenes verticales y los formatos. La resolucion, la
orientacion y el peso varian entre registros porque las fotografias provienen de telefonos de
distintos modelos y calidades, y estas propiedades condicionan el preprocesamiento porque cualquier
modelo de vision requiere entradas de tamano uniforme.

In [ ]:
tablas.resumen_numerico(datos, ["ancho", "alto", "aspecto", "megapixeles", "peso_kb"])

In [ ]:
resolucion = datos["ancho"].astype(str) + "x" + datos["alto"].astype(str)
resolucion.value_counts().head(10)

In [ ]:
print(f"proporcion de imagenes verticales: {datos['vertical'].mean():.4f}")
datos["formato"].value_counts()

In [ ]:
datos.groupby("temporada", observed=True).agg(
    mediana_megapixeles=("megapixeles", "median"),
    mediana_peso_kb=("peso_kb", "median"),
    mediana_aspecto=("aspecto", "median"),
    prop_vertical=("vertical", "mean"),
).round(3)

In [ ]:
_ = graficos.caja_por_categoria(
    datos, "temporada", "megapixeles", "Megapixeles por temporada",
    nombre_archivo="03_caja_megapixeles_temporada",
)

## Composicion de la resolucion por temporada

Esta seccion busca el hallazgo tecnico central del cuaderno. Se cruza la temporada contra la
resolucion, reteniendo las combinaciones mas frecuentes, y se complementa con el porcentaje de
imagenes de cada temporada en tres bandas de megapixeles, cuyos limites se fijan tras inspeccionar la
distribucion real.

In [ ]:
datos_resolucion = datos.assign(resolucion=datos["ancho"].astype(str) + "x" + datos["alto"].astype(str))
tabla_resolucion = datos_resolucion.pivot_table(
    index="temporada", columns="resolucion", values="ruta_relativa", aggfunc="count", fill_value=0, observed=True
)
columnas_frecuentes = tabla_resolucion.columns[tabla_resolucion.sum() >= 200]
tabla_resolucion[columnas_frecuentes].T

In [ ]:
datos["megapixeles"].describe()

In [ ]:
bandas_megapixeles = pd.cut(
    datos["megapixeles"], bins=[-0.01, 0.5, 5, 100],
    labels=["baja (<0.5 MP)", "media (0.5-5 MP)", "alta (>5 MP)"],
)
(pd.crosstab(datos["temporada"], bandas_megapixeles, normalize="index") * 100).round(2)

La composicion de resolucion no es uniforme entre temporadas. Las dos temporadas mas antiguas,
LR2020 y SR2020, son homogeneas: la totalidad de sus imagenes esta en 640x480 y en la banda baja de
megapixeles. La temporada mas reciente, SR2021, es una mezcla heterogenea: cerca del 61 por ciento
en la banda baja y casi el 38 por ciento en la banda alta, con multiples resoluciones de varios
megapixeles que no aparecen en las temporadas previas. LR2021 es una etapa de transicion. Esto
indica que el proceso de preparacion de las imagenes cambio entre temporadas, un cambio anterior a
cualquier consideracion agronomica que por si solo puede explicar parte de la perdida de desempeno
entre temporadas que motiva el desafio.

## Dispositivos de captura

Se revisa cuantas imagenes conservan metadatos EXIF de la camara y como se distribuyen las marcas y
modelos declarados.

In [ ]:
datos["tiene_exif"] = datos["marca"].notna()
print(f"proporcion con marca de camara declarada: {datos['marca'].notna().mean():.4f}")
display(datos["marca"].value_counts(dropna=False).head(10))
tablas.tabla_contingencia(datos, "temporada", "tiene_exif")

Ninguna imagen conserva metadatos EXIF de marca ni modelo. Es un resultado negativo pero relevante:
descarta atribuir las diferencias visuales entre temporadas a modelos concretos de telefono, y
obliga a apoyar el analisis unicamente en el contenido de los pixeles.

## Atributos fotometricos

Se resumen los atributos que describen cada imagen. El brillo y el contraste describen la exposicion;
la nitidez detecta imagenes desenfocadas o con movimiento; la saturacion describe la intensidad del
color; y el exceso de verde es la medida mas directamente relacionada con el estado del cultivo,
porque un cultivo afectado por sequia pierde verdor.

In [ ]:
tablas.resumen_numerico(
    datos, ["brillo", "contraste", "nitidez", "saturacion", "exceso_verde", "prop_sobreexpuesta", "prop_subexpuesta"]
)

In [ ]:
_ = graficos.histograma(datos["brillo"], "Distribucion del brillo", "brillo", nombre_archivo="03_histograma_brillo")

In [ ]:
_ = graficos.histograma(
    datos["exceso_verde"], "Distribucion del exceso de verde", "exceso de verde",
    nombre_archivo="03_histograma_exceso_verde",
)

In [ ]:
_ = graficos.histograma(
    datos["nitidez"], "Distribucion de la nitidez", "nitidez", bins=50, nombre_archivo="03_histograma_nitidez"
)

## Calidad de captura

Se considera desenfocada una imagen con nitidez por debajo de `UMBRAL_NITIDEZ`, y mal expuesta
aquella con mas de una quinta parte de sus pixeles en los extremos de la escala. Se calcula la
proporcion en cada categoria, en general y por temporada, y se muestran las imagenes de menor
nitidez.

In [ ]:
datos["desenfocada"] = datos["nitidez"] < UMBRAL_NITIDEZ
datos["mal_expuesta"] = (datos["prop_sobreexpuesta"] + datos["prop_subexpuesta"]) > 0.2
print(f"umbral de nitidez: {UMBRAL_NITIDEZ}")
print(f"proporcion desenfocada: {datos['desenfocada'].mean():.4f}")
print(f"proporcion mal expuesta: {datos['mal_expuesta'].mean():.4f}")
datos.groupby("temporada", observed=True)[["desenfocada", "mal_expuesta"]].mean().round(4)

In [ ]:
mas_borrosas = datos.nsmallest(10, "nitidez")
_ = graficos.mosaico_imagenes(
    mas_borrosas["ruta_relativa"].tolist(),
    [f"nitidez={valor:.0f}" for valor in mas_borrosas["nitidez"]],
    "Diez imagenes de menor nitidez", columnas=5, nombre_archivo="03_mosaico_desenfocadas",
)

La calidad de captura no es homogenea: el desenfoque se concentra en SR2021, que pasa del practicamente
nulo de LR2020 a cerca del 3.7 por ciento. Esto es coherente con la heterogeneidad de resolucion de
esa misma temporada: el cambio en el proceso de captura de SR2021 introdujo tanto resoluciones
dispares como una mayor proporcion de imagenes desenfocadas. La mala exposicion, en cambio, se
mantiene baja y estable entre temporadas.

## Imagenes repetidas

La huella perceptual permite detectar contenido repetido aunque los archivos difieran en compresion o
tamano: dos imagenes pertenecen al mismo grupo visual cuando la distancia de Hamming entre sus huellas
no supera el maximo configurado. Las huellas degeneradas, propias de imagenes de color casi uniforme,
se agrupan entre si sin que su contenido sea realmente el mismo, por lo que se excluyen antes de
agrupar.

In [ ]:
degenerados = caracteristicas.detectar_hashes_degenerados(datos["hash_perceptual"])
print(f"hashes degenerados excluidos: {int(degenerados.sum())}")
validas = datos[~degenerados].copy()
validas["grupo_visual"] = caracteristicas.agrupar_duplicados(validas["hash_perceptual"].tolist())
display(caracteristicas.resumen_duplicados(validas).T)
print(f"imagenes con huella identica a otra: {int(validas['hash_perceptual'].duplicated(keep=False).sum())}")

## Solapamiento visual entre particiones

Sobre los grupos visuales, se cuenta cuantos contienen imagenes de ambas particiones y cuantas
imagenes estan involucradas en esos grupos mixtos.

In [ ]:
grupos_por_particiones = validas.groupby("grupo_visual")["particion"].nunique()
grupos_mixtos = grupos_por_particiones[grupos_por_particiones > 1].index
print(f"grupos visuales con imagenes de ambas particiones: {len(grupos_mixtos)}")
print(f"imagenes involucradas en grupos mixtos: {int(validas['grupo_visual'].isin(grupos_mixtos).sum())}")

In [ ]:
tamanos_mixtos = (
    validas[validas["grupo_visual"].isin(grupos_mixtos)].groupby("grupo_visual").size().sort_values(ascending=False)
)
for grupo in tamanos_mixtos.head(2).index:
    miembros = validas[validas["grupo_visual"] == grupo]
    subtitulos = [f"{particion} mag={magnitud}" for particion, magnitud in zip(miembros["particion"], miembros["magnitud"])]
    _ = graficos.mosaico_imagenes(
        miembros["ruta_relativa"].tolist(), subtitulos, f"Grupo visual {grupo}", columnas=4,
        nombre_archivo=f"03_mosaico_grupo_{grupo}",
    )

La duplicacion visual es marginal: solo unas decenas de imagenes forman grupos mixtos entre
entrenamiento y prueba. Frente a la fuga por campo documentada en el cuaderno 02, donde alrededor del
80 por ciento de los registros cae en campos compartidos, la fuga por contenido duplicado es
despreciable. El mecanismo de fuga relevante para el diseno de una validacion es el campo, no la
imagen repetida.

## Contenido visual segun la etiqueta

Se muestran ejemplos representativos por tipo de dano y ejemplos del subconjunto de sequia ordenados
por magnitud creciente, para juzgar si la diferencia entre categorias resulta apreciable a simple
vista.

In [ ]:
muestras_dano = []
for nivel in [d for d in ORDEN_DANOS if (entrenamiento["dano"] == d).any()]:
    grupo = entrenamiento[entrenamiento["dano"] == nivel]
    muestras_dano.append(grupo.sample(min(3, len(grupo)), random_state=7))
ejemplos_dano = pd.concat(muestras_dano)
_ = graficos.mosaico_imagenes(
    ejemplos_dano["ruta_relativa"].tolist(),
    [f"{dano} mag={magnitud}" for dano, magnitud in zip(ejemplos_dano["dano"], ejemplos_dano["magnitud"])],
    "Ejemplos por tipo de dano", columnas=6, nombre_archivo="03_mosaico_por_dano",
)

In [ ]:
sequia_train = entrenamiento[entrenamiento["dano"] == "DR"]
muestras_magnitud = []
for nivel in sorted(sequia_train["magnitud"].dropna().unique()):
    grupo = sequia_train[sequia_train["magnitud"] == nivel]
    muestras_magnitud.append(grupo.sample(min(2, len(grupo)), random_state=7))
ejemplos_magnitud = pd.concat(muestras_magnitud)
_ = graficos.mosaico_imagenes(
    ejemplos_magnitud["ruta_relativa"].tolist(),
    [f"mag={magnitud}" for magnitud in ejemplos_magnitud["magnitud"]],
    "Ejemplos de sequia por magnitud creciente", columnas=6, nombre_archivo="03_mosaico_por_magnitud",
)

La diferencia entre categorias y entre niveles de magnitud no es evidente a simple vista: muchas
imagenes de sequia severa se parecen a imagenes sanas, lo que anticipa la dificultad de estimar la
magnitud visualmente y justifica un modelo capaz de captar textura y contexto.

## Relacion entre los atributos visuales y la magnitud

Se usa la correlacion de Spearman en vez de Pearson porque la magnitud es discreta y ordinal y la
relacion no tiene por que ser lineal. Se calcula sobre el subconjunto de entrenamiento.

In [ ]:
atributos_foto = [
    "exceso_verde", "brillo", "contraste", "nitidez", "saturacion", "prop_sobreexpuesta", "prop_subexpuesta", "megapixeles"
]
tablas.correlacion_contra_objetivo(entrenamiento, atributos_foto, "magnitud", metodo="spearman")

In [ ]:
entrenamiento.groupby("magnitud", observed=True)[["exceso_verde", "brillo", "nitidez"]].mean().round(4)

In [ ]:
_ = graficos.caja_por_categoria(
    entrenamiento, "magnitud", "exceso_verde", "Exceso de verde por nivel de magnitud",
    nombre_archivo="03_caja_verde_magnitud",
)

El exceso de verde presenta la relacion mas fuerte con la magnitud, con una correlacion de Spearman
cercana a -0.25, y esa relacion es monotona: el verdor promedio decae de forma sostenida a medida que
la magnitud crece, desde un valor claramente positivo en magnitud cero hasta un valor negativo en
magnitud cien. Ningun otro atributo alcanza una asociacion comparable. Que la relacion sea solo
moderada indica que un indicador simple de verdor no basta para resolver el problema y que se requiere
un modelo capaz de captar textura y contexto.

## Relacion entre los atributos visuales y el tipo de dano

In [ ]:
datos.groupby("dano", observed=True)[["exceso_verde", "brillo", "saturacion"]].mean().round(4)

In [ ]:
_ = graficos.caja_por_categoria(
    datos, "dano", "exceso_verde", "Exceso de verde por tipo de dano", nombre_archivo="03_caja_verde_dano"
)

## Diferencias entre temporadas en las imagenes

Si las condiciones de captura cambian entre temporadas, un modelo ajustado sobre las primeras aprende
parte de esas condiciones y falla al aplicarse a la siguiente. Se comparan los atributos por temporada
y se contrastan las tres primeras temporadas contra la ultima disponible mediante la prueba de
Kolmogorov y Smirnov.

In [ ]:
datos.groupby("temporada", observed=True)[atributos_foto].mean().round(3)

In [ ]:
for atributo, nombre_figura in (
    ("exceso_verde", "03_caja_exceso_verde_temporada"),
    ("brillo", "03_caja_brillo_temporada"),
    ("nitidez", "03_caja_nitidez_temporada"),
):
    _ = graficos.caja_por_categoria(datos, "temporada", atributo, f"{atributo} por temporada", nombre_archivo=nombre_figura)

In [ ]:
ultima_temporada = "SR2021"
primeras_temporadas = datos[datos["temporada"] != ultima_temporada]
imagenes_ultima = datos[datos["temporada"] == ultima_temporada]
tablas.comparar_variables_numericas(primeras_temporadas, imagenes_ultima, atributos_foto, etiquetas=("primeras", "SR2021"))

El mayor desplazamiento entre las primeras temporadas y SR2021 esta en los megapixeles, con un
estadistico de Kolmogorov y Smirnov superior a 0.55: SR2021 tiene una resolucion mucho mayor. Le
siguen la saturacion, la nitidez y el exceso de verde, todos menores en SR2021. El desplazamiento
fotometrico entre temporadas esta encabezado por la propiedad tecnica que ya identificamos en la
composicion de resolucion, lo que refuerza que el cambio de captura, y no solo el estado del cultivo,
separa a SR2021 de las temporadas anteriores.

## Diferencias entre entrenamiento y prueba en las imagenes

Se repite la comparacion de Kolmogorov y Smirnov entre entrenamiento y prueba.

In [ ]:
tablas.comparar_variables_numericas(entrenamiento, prueba, atributos_foto, etiquetas=("train", "test"))

Las diferencias entre entrenamiento y prueba son minimas: todos los estadisticos de Kolmogorov y
Smirnov quedan por debajo de 0.02 y ninguno es significativo. Contrastadas con el fuerte
desplazamiento entre temporadas de la seccion anterior, confirman que el desplazamiento relevante
ocurre entre temporadas y no es visible al evaluar sobre la particion oficial de la competencia, en
linea con lo que el cuaderno 02 encontro sobre las etiquetas.

## Cierre de la etapa

Los atributos visuales completan el diagnostico iniciado con las etiquetas: la resolucion y la calidad
de captura cambian de forma marcada en SR2021, el exceso de verde mantiene una relacion monotona pero
moderada con la magnitud, la duplicacion visual es despreciable frente a la fuga por campo, y el
desplazamiento entre temporadas se manifiesta con claridad en las imagenes aunque no entre
entrenamiento y prueba. El informe final consolida los hallazgos de los tres cuadernos y define los
siguientes pasos del proyecto.